# Qwen3-30B-A3B MI355X — linear-op timing, settle-8 collection (2026-09-23)

**A separate notebook for a separate, newer dataset.** Does not modify or replace
`linear_ops_run_groups_and_stats.ipynb` (2026-09-15/16) or
`linear_ops_dense_fixed_run_groups_and_stats.ipynb` (2026-09-22, job 21483/21519) — both untouched.

## Collection details

- **Collected:** 2026-09-23 09:49-10:04 UTC, Slurm job 21539 (`q3-dense-workbacklog-s8`), node
  `amd-mi355x-1`, dense grid (13,308 rows = 3,327 `num_tokens` values x TP {1,2,4,8}, same grid as
  every prior collection).
- **What changed vs. job 21519** (`data/profiling_dense_fixed_workbacklog/`): **8 untimed settle
  forwards after the GEMM work-backlog spin, instead of 3** (`FRONTIER_LINEAR_SETTLE_STEPS=8`).
  Everything else — `FRONTIER_GPU_BACKLOG_KIND=gemm_alloc`, `DEBUG_CLR_MAX_BATCH_SIZE=1000000`,
  25 timed forwards per shape — is unchanged from 21519.
- **Why:** job 21519 left a ~1% transient over timed runs 1-4 at every TP and a slow 0.7% decline
  over the block at TP1 (see its README and
  `13_run_position_anomalies_root_cause.md`, Effect 1). This run tests whether five more untimed
  forwards after the backlog absorb that transient.
- **Source path (cluster):**
  `/opt/shared/frontier-qwen3-profiling/Frontier/data/profiling_dense_fixed_workbacklog_settle8/compute/mi355x/qwen3-a3b-30b-moe/linear_op.csv`
- **Code state:** worktree commit `3a8eb55` (verified identical on the cluster before submission).
- **Verified in this file:** `settle_steps` is 8 on every row, `backlog_kind` is `gemm_alloc` on
  every row, `attn_rope_impl` is `vllm_kernel` on every row, `sclk_mhz_backlog_start` >= 2.2 GHz on
  13,303/13,308 rows, and `gpu_backlog_ms_actual / gpu_backlog_ms` has a median of 1.0001 (queue
  backpressure hit its target).

## What's in this notebook

Two plots per (op, TP), using this file's own aggregate columns (`time_stats.<op>.{median,mean,min,max}`)
directly — no run-group splitting (that analysis lives in the two prior notebooks):

1. **Median only** — a single line, median time vs. `num_tokens`.
2. **Min-max envelope + mean (solid) + median (dashed)** — same shape as the "overall summary"
   plot in the prior two notebooks.

Y-axis is log-scale (GEMM/norm time spans ~3-90x across the `num_tokens` grid). `emb`,
`input_layernorm`, `post_attention_layernorm` are TP=1 only, by design. `forward_gpu_span` (whole-
forward device wall span) gets the same two plots even though it isn't an attention-layer op.

Vertical lines on `attn_pre_proj`/`attn_post_proj` mark a confirmed GEMM kernel/tile-config change
at that `num_tokens` (full-grid `rocprofv3` trace, 2026-09-22), colored by a structural severity
score computed from tile geometry alone (grey/orange/red = low/medium/high expected impact) — see
the comparison notebook's intro for the exact rule. Same boundaries as the other two notebooks:
kernel selection is a property of the shape and the ROCm/hipBLASLt stack, not of which collection
this data came from.

In [1]:
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = "/tmp/claude-1001/-home-dn-amd-playground/dedce0c4-4547-4649-84a2-ad89dca651aa/scratchpad/settle8"
lin = pd.read_csv(f"{DATA_DIR}/linear_op.csv", low_memory=False, float_precision="round_trip")

OPS = ["attn_pre_proj", "attn_post_proj", "attn_rope",
       "input_layernorm", "post_attention_layernorm", "emb", "forward_gpu_span"]

LINE_COLOR = "#2a78d6"
ENVELOPE_FILL = "rgba(42, 120, 214, 0.18)"

SURFACE = "#fcfcfb"
GRID = "#e1e0d9"
AXIS_LINE = "#c3c2b7"
TEXT_PRIMARY = "#0b0b0b"

# Kernel-change line color by expected-impact severity - a structural score from tile geometry
# alone (never from the observed timing). Same scale/rule as the other two notebooks.
KERNEL_SEVERITY_COLOR = {
    "low": "rgba(150, 150, 150, 0.30)",
    "medium": "rgba(237, 165, 0, 0.45)",
    "high": "rgba(211, 47, 47, 0.55)",
}


In [2]:
KERNEL_CHANGE_BOUNDARIES = {
    1: [[17, 'high'], [32, 'high'], [33, 'high'], [48, 'low'], [49, 'high'], [64, 'high'], [65, 'high'], [81, 'high'], [96, 'medium'], [97, 'low'], [128, 'low'], [129, 'high'], [145, 'low'], [160, 'high'], [161, 'high'], [176, 'high'], [177, 'high'], [192, 'high'], [193, 'high'], [208, 'high'], [209, 'high'], [224, 'high'], [225, 'high'], [240, 'high'], [241, 'high'], [256, 'high'], [257, 'high'], [289, 'low'], [305, 'low'], [321, 'high'], [353, 'low'], [385, 'high'], [481, 'low'], [513, 'high'], [545, 'low'], [577, 'high'], [641, 'low'], [673, 'medium'], [705, 'high'], [769, 'high'], [801, 'high'], [849, 'low'], [865, 'low'], [897, 'high'], [961, 'low'], [1025, 'high'], [1057, 'low'], [1089, 'low'], [1153, 'high'], [1217, 'high'], [1281, 'high'], [1345, 'high'], [1409, 'low'], [1441, 'low'], [1473, 'low'], [1537, 'high'], [1601, 'high'], [1793, 'high'], [1825, 'low'], [1857, 'low'], [1873, 'low'], [1921, 'high'], [2017, 'low'], [2056, 'high'], [2120, 'high'], [2312, 'high'], [2440, 'low'], [2568, 'high'], [2696, 'low'], [2824, 'high'], [2952, 'high'], [3080, 'medium'], [3208, 'high'], [3272, 'high'], [3336, 'high'], [3368, 'low'], [3400, 'low'], [3464, 'low'], [3496, 'low'], [3592, 'medium'], [3608, 'low'], [3656, 'high'], [3784, 'high'], [3880, 'high'], [4040, 'low'], [4104, 'high'], [4232, 'low'], [4264, 'high'], [4360, 'low'], [4408, 'low'], [4424, 'high'], [4488, 'low'], [4616, 'low'], [4680, 'low'], [4712, 'low'], [4744, 'low'], [4808, 'high'], [4872, 'low'], [5128, 'medium'], [5192, 'high'], [5384, 'medium'], [5448, 'low'], [5576, 'high'], [5608, 'low'], [5640, 'low'], [5768, 'high'], [5896, 'high'], [6152, 'high'], [6408, 'high'], [6536, 'low'], [6664, 'low'], [6728, 'high'], [8072, 'low'], [8208, 'low'], [8720, 'low'], [8752, 'high'], [8976, 'high'], [9232, 'low'], [9744, 'high'], [9808, 'high'], [10000, 'high'], [10256, 'low'], [10768, 'medium'], [10896, 'low'], [11056, 'low'], [11344, 'high'], [11440, 'low'], [11792, 'low'], [11856, 'low'], [12304, 'high'], [13072, 'low'], [13616, 'low'], [13648, 'low'], [16336, 'low']],
    2: [[17, 'high'], [32, 'high'], [33, 'high'], [48, 'high'], [49, 'high'], [64, 'high'], [65, 'high'], [81, 'high'], [96, 'high'], [97, 'high'], [112, 'low'], [113, 'high'], [128, 'high'], [129, 'high'], [144, 'high'], [145, 'high'], [160, 'high'], [161, 'high'], [176, 'low'], [177, 'low'], [192, 'low'], [193, 'high'], [240, 'low'], [241, 'low'], [256, 'high'], [257, 'high'], [321, 'low'], [337, 'high'], [385, 'high'], [401, 'low'], [449, 'low'], [513, 'high'], [577, 'low'], [609, 'low'], [641, 'high'], [673, 'high'], [705, 'low'], [769, 'high'], [801, 'high'], [897, 'high'], [1025, 'high'], [1153, 'low'], [1217, 'high'], [1281, 'high'], [1345, 'high'], [1537, 'high'], [1601, 'low'], [1633, 'low'], [1697, 'low'], [1729, 'low'], [1793, 'high'], [1825, 'low'], [1921, 'low'], [2056, 'high'], [2120, 'low'], [2184, 'low'], [2216, 'high'], [2248, 'low'], [2312, 'high'], [2408, 'high'], [2440, 'high'], [2568, 'high'], [2696, 'high'], [2824, 'high'], [2888, 'high'], [2952, 'high'], [3016, 'low'], [3080, 'high'], [3144, 'low'], [3208, 'high'], [3240, 'low'], [3272, 'low'], [3336, 'low'], [3368, 'low'], [3464, 'low'], [3592, 'high'], [3688, 'high'], [3752, 'high'], [3784, 'high'], [3848, 'high'], [4008, 'low'], [4040, 'low'], [4104, 'high'], [4328, 'low'], [4424, 'low'], [4488, 'low'], [4808, 'low'], [4872, 'low'], [5128, 'high'], [5384, 'high'], [5448, 'low'], [6152, 'high'], [6408, 'medium'], [6536, 'high'], [6728, 'high'], [6792, 'medium'], [6808, 'low'], [6824, 'low'], [6984, 'low'], [7176, 'high'], [7304, 'low'], [8072, 'medium'], [8168, 'low'], [8208, 'high'], [8560, 'low'], [8656, 'low'], [8720, 'high'], [8752, 'low'], [8848, 'low'], [8976, 'low'], [9232, 'high'], [9424, 'low'], [9552, 'low'], [9808, 'medium'], [10000, 'low'], [10256, 'high'], [10768, 'high'], [10896, 'low'], [11024, 'low'], [11152, 'low'], [11280, 'low'], [11440, 'high'], [11472, 'low'], [11536, 'low'], [11920, 'low'], [12304, 'high'], [12816, 'low'], [13072, 'low'], [13328, 'low'], [13616, 'low'], [13840, 'high'], [14352, 'high'], [16336, 'low']],
    4: [[16, 'high'], [32, 'high'], [33, 'high'], [49, 'high'], [64, 'high'], [65, 'high'], [81, 'low'], [96, 'high'], [97, 'high'], [128, 'high'], [129, 'high'], [160, 'high'], [161, 'high'], [176, 'high'], [177, 'high'], [192, 'high'], [193, 'high'], [208, 'low'], [209, 'low'], [224, 'low'], [225, 'low'], [240, 'low'], [241, 'low'], [256, 'high'], [257, 'high'], [321, 'low'], [337, 'high'], [385, 'high'], [513, 'high'], [577, 'low'], [609, 'high'], [641, 'low'], [673, 'high'], [769, 'high'], [801, 'medium'], [817, 'low'], [849, 'low'], [897, 'high'], [1025, 'high'], [1089, 'low'], [1153, 'low'], [1217, 'high'], [1281, 'high'], [1345, 'high'], [1409, 'low'], [1537, 'high'], [1601, 'high'], [1761, 'low'], [1857, 'low'], [2056, 'high'], [2216, 'low'], [2312, 'medium'], [2408, 'high'], [2440, 'low'], [2568, 'high'], [2696, 'low'], [2728, 'medium'], [2952, 'low'], [3080, 'high'], [3144, 'low'], [3176, 'low'], [3208, 'high'], [3240, 'low'], [3272, 'high'], [3336, 'low'], [3368, 'low'], [3464, 'high'], [3592, 'high'], [3656, 'low'], [3688, 'medium'], [3720, 'high'], [3752, 'low'], [3848, 'low'], [4008, 'high'], [4104, 'high'], [4264, 'low'], [4360, 'low'], [4616, 'low'], [4744, 'low'], [4808, 'high'], [4872, 'low'], [5000, 'low'], [5128, 'high'], [5192, 'low'], [5384, 'medium'], [5448, 'high'], [5608, 'high'], [5640, 'low'], [5720, 'low'], [5768, 'low'], [6152, 'high'], [6408, 'low'], [6536, 'low'], [6728, 'high'], [6792, 'low'], [6920, 'high'], [7176, 'high'], [7208, 'low'], [7304, 'low'], [7368, 'low'], [7560, 'low'], [7688, 'low'], [7944, 'low'], [8072, 'low'], [8168, 'high'], [8208, 'high'], [8336, 'low'], [8464, 'high'], [8912, 'low'], [8976, 'low'], [9040, 'low'], [9232, 'high'], [9456, 'low'], [9808, 'high'], [10256, 'high'], [10320, 'low'], [10384, 'low'], [10768, 'high'], [10896, 'low'], [11344, 'low'], [11440, 'low'], [11536, 'low'], [11664, 'low'], [11792, 'low'], [11920, 'low'], [12176, 'low'], [12304, 'high'], [12816, 'high'], [13072, 'high'], [13616, 'high'], [13648, 'low'], [13840, 'low'], [13968, 'low'], [14032, 'low'], [14352, 'high'], [14608, 'low'], [14704, 'low'], [14736, 'low'], [16336, 'low']],
    8: [[16, 'high'], [32, 'high'], [33, 'high'], [64, 'high'], [65, 'high'], [81, 'high'], [96, 'high'], [97, 'high'], [112, 'low'], [113, 'low'], [128, 'high'], [129, 'high'], [144, 'high'], [145, 'high'], [160, 'high'], [161, 'high'], [225, 'high'], [257, 'high'], [321, 'low'], [337, 'high'], [385, 'low'], [513, 'medium'], [641, 'high'], [673, 'high'], [705, 'low'], [769, 'low'], [801, 'high'], [897, 'low'], [1025, 'low'], [1089, 'high'], [1153, 'high'], [1217, 'high'], [1281, 'low'], [1345, 'high'], [1361, 'high'], [1537, 'high'], [1633, 'low'], [1697, 'low'], [1793, 'high'], [1921, 'low'], [2056, 'high'], [2120, 'high'], [2248, 'low'], [2312, 'high'], [2440, 'low'], [2568, 'high'], [2696, 'high'], [2728, 'high'], [2824, 'low'], [2888, 'low'], [2952, 'high'], [3016, 'low'], [3080, 'high'], [3208, 'high'], [3240, 'low'], [3368, 'low'], [3464, 'low'], [3592, 'low'], [3656, 'high'], [3752, 'low'], [3784, 'high'], [3848, 'low'], [4008, 'high'], [4040, 'low'], [4104, 'high'], [4264, 'high'], [4360, 'high'], [4456, 'high'], [4488, 'high'], [5128, 'high'], [5384, 'low'], [5448, 'low'], [6152, 'high'], [6408, 'low'], [6536, 'low'], [6728, 'low'], [6792, 'low'], [6808, 'low'], [6824, 'low'], [7176, 'high'], [8072, 'low'], [8208, 'high'], [8464, 'high'], [8720, 'low'], [8752, 'high'], [8848, 'high'], [9232, 'high'], [9296, 'low'], [9424, 'high'], [9552, 'low'], [9808, 'high'], [10000, 'low'], [10256, 'high'], [10768, 'high'], [10896, 'low'], [11024, 'low'], [11152, 'high'], [11440, 'high'], [11664, 'high'], [11920, 'high'], [12304, 'high'], [12816, 'low'], [13072, 'low'], [13616, 'low'], [13840, 'low'], [14352, 'high'], [16336, 'low']],
}

def add_kernel_change_lines(fig, op, tp):
    """Draw a thin vertical line at every num_tokens where attn_pre_proj/attn_post_proj's GEMM
    kernel selection changes (confirmed by kernel trace, not inferred from timing), colored by
    the structural severity score (grey/orange/red = low/medium/high expected impact)."""
    if op not in ("attn_pre_proj", "attn_post_proj"):
        return
    for tok, severity in KERNEL_CHANGE_BOUNDARIES.get(tp, []):
        fig.add_vline(x=tok, line=dict(color=KERNEL_SEVERITY_COLOR[severity], width=1, dash="dot"))


print(lin.shape, "rows x cols")
print("num_tokens grid:", lin["num_tokens"].nunique(), "values,",
      lin["num_tokens"].min(), "to", lin["num_tokens"].max())
print("settle_steps:", lin["settle_steps"].unique().tolist())
print("backlog_kind:", lin["backlog_kind"].unique().tolist())
print("attn_rope_impl:", lin["attn_rope_impl"].unique().tolist())
print("rows with sclk_mhz_backlog_start >= 2200 MHz:",
      int((lin["sclk_mhz_backlog_start"] >= 2200).sum()), "/", len(lin))
print("backlog closure (gpu_backlog_ms_actual / gpu_backlog_ms) median:",
      (lin["gpu_backlog_ms_actual"] / lin["gpu_backlog_ms"]).median())


(13308, 150) rows x cols
num_tokens grid: 3327 values, 1 to 16384
settle_steps: [8]
backlog_kind: ['gemm_alloc']
attn_rope_impl: ['vllm_kernel']
rows with sclk_mhz_backlog_start >= 2200 MHz: 13303 / 13308
backlog closure (gpu_backlog_ms_actual / gpu_backlog_ms) median: 1.000108164504074


In [3]:
def tp_values_with_data(df, op):
    """TP values (sorted) for which `op` has a non-null aggregate — excludes TP>1
    for the TP1-only ops (input_layernorm, post_attention_layernorm, emb)."""
    col = f"time_stats.{op}.mean"
    present = df.loc[df[col].notna(), "num_tensor_parallel_workers"].unique()
    return sorted(int(t) for t in present)


def op_stats(df, op, tp):
    """(num_tokens, mean, median, min, max) for this op/TP, straight from the CSV's
    own aggregate columns — no run-group splitting, no outlier filtering."""
    stat_cols = {s: f"time_stats.{op}.{s}" for s in ["mean", "median", "min", "max"]}
    d = df[(df.num_tensor_parallel_workers == tp) & df[stat_cols["mean"]].notna()]
    d = d[["num_tokens", *stat_cols.values()]].rename(columns={v: k for k, v in stat_cols.items()})
    return d.sort_values("num_tokens")


def plot_median_only(df, op, tp):
    d = op_stats(df, op, tp)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.num_tokens, y=d["median"], mode="lines", name="Median",
                              line=dict(color=LINE_COLOR, width=2)))
    fig.update_xaxes(title_text="num_tokens (shape)", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_yaxes(title_text="median time (ms), log scale", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_layout(
        title=f"{op} (TP {tp}) — median time over timed runs",
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=TEXT_PRIMARY),
        height=450, width=900,
    )
    add_kernel_change_lines(fig, op, tp)
    return fig


def plot_envelope_mean_median(df, op, tp):
    d = op_stats(df, op, tp)
    fig = go.Figure()
    # Min-max envelope: invisible min line, then max line filled back down to it.
    fig.add_trace(
        go.Scatter(x=d.num_tokens, y=d["min"], mode="lines",
                   line=dict(width=0), hoverinfo="skip", showlegend=False),
    )
    fig.add_trace(
        go.Scatter(x=d.num_tokens, y=d["max"], mode="lines",
                   line=dict(width=0), fill="tonexty", fillcolor=ENVELOPE_FILL,
                   name="Min–max range"),
    )
    # Mean: solid. Median: dashed. Same hue — distinguished by line style.
    fig.add_trace(
        go.Scatter(x=d.num_tokens, y=d["mean"], mode="lines", name="Mean",
                   line=dict(color=LINE_COLOR, width=2, dash="solid")),
    )
    fig.add_trace(
        go.Scatter(x=d.num_tokens, y=d["median"], mode="lines", name="Median",
                   line=dict(color=LINE_COLOR, width=2, dash="dash")),
    )
    fig.update_xaxes(title_text="num_tokens (shape)", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_yaxes(title_text="time (ms), log scale", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_layout(
        title=f"{op} (TP {tp}) — mean / median (dashed) over timed runs, min–max envelope",
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=TEXT_PRIMARY),
        height=500, width=900,
    )
    add_kernel_change_lines(fig, op, tp)
    return fig


## `attn_pre_proj`

In [4]:
for tp in tp_values_with_data(lin, "attn_pre_proj"):
    plot_median_only(lin, "attn_pre_proj", tp).show()

In [5]:
for tp in tp_values_with_data(lin, "attn_pre_proj"):
    plot_envelope_mean_median(lin, "attn_pre_proj", tp).show()

## `attn_post_proj`

In [6]:
for tp in tp_values_with_data(lin, "attn_post_proj"):
    plot_median_only(lin, "attn_post_proj", tp).show()

In [7]:
for tp in tp_values_with_data(lin, "attn_post_proj"):
    plot_envelope_mean_median(lin, "attn_post_proj", tp).show()

## `attn_rope`

In [8]:
for tp in tp_values_with_data(lin, "attn_rope"):
    plot_median_only(lin, "attn_rope", tp).show()

In [9]:
for tp in tp_values_with_data(lin, "attn_rope"):
    plot_envelope_mean_median(lin, "attn_rope", tp).show()

## `input_layernorm`

In [10]:
for tp in tp_values_with_data(lin, "input_layernorm"):
    plot_median_only(lin, "input_layernorm", tp).show()

In [11]:
for tp in tp_values_with_data(lin, "input_layernorm"):
    plot_envelope_mean_median(lin, "input_layernorm", tp).show()

## `post_attention_layernorm`

In [12]:
for tp in tp_values_with_data(lin, "post_attention_layernorm"):
    plot_median_only(lin, "post_attention_layernorm", tp).show()

In [13]:
for tp in tp_values_with_data(lin, "post_attention_layernorm"):
    plot_envelope_mean_median(lin, "post_attention_layernorm", tp).show()

## `emb`

In [14]:
for tp in tp_values_with_data(lin, "emb"):
    plot_median_only(lin, "emb", tp).show()

In [15]:
for tp in tp_values_with_data(lin, "emb"):
    plot_envelope_mean_median(lin, "emb", tp).show()

## `forward_gpu_span`

Whole-forward device wall span (start of the first queued kernel to the end of the last), not one attention-layer op — included here because it shares the same `time_stats.*` schema as every other op.

In [16]:
for tp in tp_values_with_data(lin, "forward_gpu_span"):
    plot_median_only(lin, "forward_gpu_span", tp).show()

In [17]:
for tp in tp_values_with_data(lin, "forward_gpu_span"):
    plot_envelope_mean_median(lin, "forward_gpu_span", tp).show()